# 00 - Setup & Configuration

**Purpose**: Environment setup, configuration loading, and initial data loading

**Outputs**:
- Loaded configuration
- Raw data loaded and ready for preprocessing
- Environment validated

---

## 1. Environment Setup

In [ ]:
# Import notebook utilities
import sys
from pathlib import Path

# Add project root to path
PROJECT_ROOT = Path.cwd().parent if 'notebooks' in str(Path.cwd()) else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project Root: {PROJECT_ROOT}")
print(f"Python Version: {sys.version}")

In [ ]:
# Check environment and imports
from src._01_setup.environment import check_environment

check_environment()
print("\n✓ Environment check passed!")

In [ ]:
# Import all required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Import notebook utilities
from notebook_utils import setup_notebook, NotebookState, display_dataframe_summary

# Import project modules
from src._01_setup import config_loader

print("✓ All imports successful")

## 2. Configuration

In [ ]:
# Setup notebook
cfg, state = setup_notebook(
    title="Setup & Configuration",
    market="germany"
)

# Display current state
state.list_keys()

In [ ]:
# Display key configuration settings
print("\n📋 Configuration Settings:")
print("=" * 80)

# Data settings
market = config_loader.get_value(cfg, 'data', 'market', default='germany')
input_dir = config_loader.get_value(cfg, 'data', 'input_dir', default='data/raw')
print(f"Market: {market}")
print(f"Input Directory: {input_dir}")

# Clustering settings
algorithm = config_loader.get_value(cfg, 'classification', 'algorithm', default='kmeans')
optimal_k = config_loader.get_value(cfg, 'classification', 'optimal_k', default=5)
print(f"\nDefault Algorithm: {algorithm}")
print(f"Optimal K: {optimal_k}")

# Feature settings
feature_set = config_loader.get_value(cfg, 'analysis', 'feature_set', default='combined')
print(f"\nFeature Set: {feature_set}")

print("=" * 80)

## 3. Load Raw Data

In [ ]:
# Locate raw data files
data_dir = PROJECT_ROOT / input_dir / market
print(f"\n📁 Looking for data in: {data_dir}")

if data_dir.exists():
    files = list(data_dir.glob('*.csv'))
    print(f"\nFound {len(files)} CSV files:")
    for f in files:
        size_mb = f.stat().st_size / 1024**2
        print(f"  - {f.name:40s} ({size_mb:6.2f} MB)")
else:
    print(f"⚠️  Directory not found: {data_dir}")

In [ ]:
# Load main data file
# Adjust filename based on your actual data structure
data_file = data_dir / 'companies_features.csv'  # Adjust this filename

if data_file.exists():
    df_raw = pd.read_csv(data_file)
    display_dataframe_summary(df_raw, "Raw Data")
    
    # Save to state
    state.save('df_raw', df_raw, metadata={'source': str(data_file)})
    
    # Display first few rows
    print("\n📊 Preview:")
    display(df_raw.head())
else:
    print(f"⚠️  File not found: {data_file}")
    print("\n💡 Tip: Check your data directory and update the filename above")

## 4. Data Quality Check

In [ ]:
# Check for missing values
if 'df_raw' in locals():
    missing = df_raw.isnull().sum()
    missing_pct = (missing / len(df_raw) * 100).round(2)
    
    df_missing = pd.DataFrame({
        'Missing Count': missing,
        'Missing %': missing_pct
    }).query('`Missing Count` > 0').sort_values('Missing %', ascending=False)
    
    if len(df_missing) > 0:
        print("\n⚠️  Missing Values Detected:")
        print("=" * 80)
        display(df_missing.head(20))
    else:
        print("\n✓ No missing values detected!")

In [ ]:
# Data types overview
if 'df_raw' in locals():
    print("\n📊 Data Types:")
    print("=" * 80)
    dtype_counts = df_raw.dtypes.value_counts()
    print(dtype_counts)
    print("\n" + "=" * 80)

## 5. Save Configuration to State

In [ ]:
# Save configuration for next notebooks
config_dict = {
    'market': market,
    'input_dir': input_dir,
    'algorithm': algorithm,
    'optimal_k': optimal_k,
    'feature_set': feature_set,
    'project_root': str(PROJECT_ROOT)
}

state.save('config', config_dict)
state.save('cfg_full', cfg)

print("\n✓ Configuration saved to state")

## 6. Summary

In [ ]:
print("\n" + "=" * 80)
print("  ✓ SETUP COMPLETE")
print("=" * 80)

state.list_keys()

print("\n📝 Next Steps:")
print("  → Open notebook: 01_Data_Preprocessing.ipynb")
print("=" * 80)